# Create H5 Files from Raw Particles
Converts each experiment's restacked particles (`particles/batch_*_restacked.mrc`) and per-particle annotation spreadsheet into one `.h5` file, with particle images under `images` and annotations as a JSON string under `annotations`. By default, particles are resized to 224 × 224 px and min-max normalized per image to uint8 [0–255].

> If the downloaded experiment directories contain `.zip` files, run `unzip_data.ipynb` first.

In [1]:
# ── Configuration ─────────────────────────────────────────────────────────────
CRYOPANDA_DIR = 'PATH_TO_cryoPANDA_DIR'   # <-- set this (one sub-directory per experiment)
H5_DIR        = 'PATH_TO_H5_DIR'          # <-- set this (output directory)
METADATA_XLSX = './metadata/per_experiment_metadata.xlsx'

EMPIAR_IDS    = None   # None → all experiments in METADATA_XLSX; or a subset, e.g. [10024, 10096]
NUM_WORKERS   = 12     # parallel processes (capped to number of experiments)

RESIZE_PARTICLES        = 224    # output box size (px); None keeps the original box size
PER_IMAGE_NORMALIZATION = True   # min-max → uint8 [0–255]; False keeps float32 (MultiH5Dataset expects uint8)

In [2]:
import os
import json
import h5py
import torch
import mrcfile
import numpy as np
import pandas as pd
from multiprocessing import Pool
from utils.resize import resize_torch

## Constants

In [3]:
# Annotation columns stored in each H5 file (the JSON keeps this column order).
# Experiments processed from micrographs carry additional micrograph / picking columns.
ANN_COLUMNS_MIC = [
    'uid', 'ImageName',
    'AngleRot', 'AngleTilt', 'AnglePsi', 'OriginXAngst', 'OriginYAngst',
    'DefocusU', 'DefocusV', 'DefocusAngle',
    'ClassNumber', '2DAlignmentClassESS', 'ClassECA',
    'MicrographName', 'MicrographShapeX', 'MicrographShapeY', 'ParticleXFrac', 'ParticleYFrac',
    'MicrographPixelSize', 'NCCScore', 'CoordinateX', 'CoordinateY', 'CoordinateYTransposed',
    'ClassResolution', '3DAlignmentError',
    'EMPIAR_ID', 'EMD_ID', 'PDB_ID',
    'Voltage', 'ElectronDose', 'SphericalAberration', 'AmplitudeContrast',
    'ImageSize', 'OriginalPixelSize', 'ImagePixelSize',
    'ReconstructionResolution', 'EMD_Resolution',
    'MolecularWeight_kDa', 'Subunits', 'MaxDiameter', 'Symmetry', 'DNA', 'RNA', 'ProteinClass',
    'SampleName', 'NumberOfParticles', 'DataSourceType', 'BadReconstruction',
]

# Experiments starting from picked particles (no micrograph columns)
ANN_COLUMNS_PARTICLES = [
    'uid', 'ImageName',
    'AngleRot', 'AngleTilt', 'AnglePsi', 'OriginXAngst', 'OriginYAngst',
    'DefocusU', 'DefocusV', 'DefocusAngle',
    'ClassNumber', '2DAlignmentClassESS', 'ClassECA',
    'ClassResolution', '3DAlignmentError',
    'EMPIAR_ID', 'EMD_ID', 'PDB_ID',
    'Voltage', 'ElectronDose', 'SphericalAberration', 'AmplitudeContrast',
    'ImageSize', 'OriginalPixelSize', 'ImagePixelSize',
    'ReconstructionResolution', 'EMD_Resolution',
    'MolecularWeight_kDa', 'Subunits', 'MaxDiameter', 'Symmetry', 'DNA', 'RNA', 'ProteinClass',
    'SampleName', 'NumberOfParticles', 'DataSourceType', 'BadReconstruction',
]

## 1 · Helper functions

In [4]:
def normalization_per_particle(p):
    """
    Min-max normalize each particle to [0, 255] and quantize to uint8.

    Parameters
    ----------
    p : torch.Tensor, shape (N, H, W)

    Returns
    -------
    torch.Tensor, shape (N, H, W), dtype uint8
    """
    p_min = p.amin(dim=(-1, -2), keepdim=True)
    p_max = p.amax(dim=(-1, -2), keepdim=True)
    p = ((p - p_min) / (p_max - p_min + 1e-8)) * 255
    return torch.round(p).to(torch.uint8)


def create_single_h5(empiar_id):
    """
    Create `<empiar_id>_particles.h5` from the restacked particles and the
    per-particle annotation spreadsheet of one experiment.

    Parameters
    ----------
    empiar_id : str
        Experiment directory name inside CRYOPANDA_DIR.

    Returns
    -------
    str  — status message
    """
    h5_path = os.path.join(H5_DIR, f'{empiar_id}_particles.h5')
    if os.path.exists(h5_path):
        return f'  [skip] {empiar_id}: already created'

    try:
        # Annotations → JSON (dict of lists, keeps column order)
        ann_path = os.path.join(CRYOPANDA_DIR, empiar_id, f'{empiar_id}_particles_annotations.xlsx')
        df_ann   = pd.read_excel(ann_path)
        try:
            df_ann = df_ann[ANN_COLUMNS_MIC]
        except KeyError:
            df_ann = df_ann[ANN_COLUMNS_PARTICLES]
        annotations_json = json.dumps(df_ann.to_dict(orient='list'))

        # Particles: restacked batches sorted by batch index (batch_<k>_restacked.mrc)
        particles_dir = os.path.join(CRYOPANDA_DIR, empiar_id, 'particles')
        batch_files   = sorted(
            (f for f in os.listdir(particles_dir) if f.endswith(('.mrc', '.mrcs'))),
            key=lambda f: int(f.split('_')[1]),
        )

        particles = []
        for bf in batch_files:
            with mrcfile.open(os.path.join(particles_dir, bf)) as mrc:
                if RESIZE_PARTICLES:
                    batch = resize_torch(mrc.data, size=(RESIZE_PARTICLES, RESIZE_PARTICLES), dtype_rt=torch.float32)
                else:
                    batch = torch.from_numpy(np.copy(mrc.data))
            if PER_IMAGE_NORMALIZATION:
                batch = normalization_per_particle(batch)
            particles.append(batch)
        particles = torch.cat(particles, dim=0)

        with h5py.File(h5_path, 'w') as f:
            f.create_dataset('images', data=particles.numpy())
            f.create_dataset('annotations', data=annotations_json)

        return f'  {empiar_id} — done ({len(particles):,} particles)'

    except Exception as exc:
        return f'  [error] {empiar_id}: {exc}'

## 2 · Create H5 files
One experiment per worker process; experiments with an existing `.h5` file are skipped, so the cell can be re-run to resume.

In [5]:
df_meta    = pd.read_excel(METADATA_XLSX)
empiar_ids = [str(i) for i in (df_meta['EMPIAR_ID'] if EMPIAR_IDS is None else EMPIAR_IDS)]
n_workers  = min(NUM_WORKERS, len(empiar_ids))
os.makedirs(H5_DIR, exist_ok=True)

print(f'Creating H5 files for {len(empiar_ids)} experiments using {n_workers} workers …')
with Pool(processes=n_workers) as pool:
    for msg in pool.imap_unordered(create_single_h5, empiar_ids):
        print(msg)
print('Done')

Creating H5 files for 2 experiments using 2 workers …
  10024 — done (43,585 particles)
  10096 — done (130,000 particles)
Done
